In [2]:
# self-attention 연산 구현
# query key vale 행렬 생성
    # 입력된 문장의 단어벡터(정적벡터)가 liner 연산을 가져서 질문, 키, 실제값 텐서 변환
import torch
import torch.nn as nn 
import math
import torch.nn.functional as F 

torch.manual_seed(42)

# 단어 3개짜리 문장 임베딩차원 4차원 q/k/v로 변환된 차원은 d_k는 2차원
seq_len = 3
embed_size = 4
d_k = 2
# 가상의 입력텐서 x
x = torch.randn(1, seq_len, embed_size)

# q k v 로 만들기 위해 선형 레이더 통과
W_Q = nn.Linear(embed_size, d_k, bias=False)
W_K = nn.Linear(embed_size, d_k, bias=False)
W_V = nn.Linear(embed_size,d_k,bias=False)

Q = W_Q(x)
K = W_K(x)
V = W_V(x)

x.shape , Q.shape, K.shape, V.shape

(torch.Size([1, 3, 4]),
 torch.Size([1, 3, 2]),
 torch.Size([1, 3, 2]),
 torch.Size([1, 3, 2]))

In [3]:
# Q K V를 이용해서 각 단어가 얼마나 강하게 참고할지 점수를 매기고 최종 문맥 벡터로 만드는과정
# K.transpose(1,2) : (배치, 단어수, 차원)  1번 축(단어수)과 2번축(차원)의 자리름 서로 바꿔줌
# (배치, 차원, 단어수)  -> dot-product 행렬 곱을 할수 있는 전치행렬이 된다
# Q @ K.transpose(1,2)

# 1&2. Q k 전치행렬 내적 후 스케일링
scores = (Q @ K.transpose(1,2)) / math.sqrt(d_k)
print(f'step 1&2 K전치행렬 크기 : {K.transpose(1,2).shape}')
print(f'step 1&2 스케일된 어텐션 점수 크기 : {scores.shape}')

# 3 softmax로 어텐션 가중치 생성(합을 1.0)
atten_weight =  F.softmax(scores,dim=-1)

print(f'step3 어텐션 가중치 행렬 : {atten_weight}')
print('~~ 들의 가중치를모두 더하면 정확히 1.0 이 됩니다.')

# 4 가중치와 value를 곱해 최종 context vector 생성
context_vector = atten_weight @ V
print(f'최종 문맥 벡터 크기 : {context_vector.shape}')

step 1&2 K전치행렬 크기 : torch.Size([1, 2, 3])
step 1&2 스케일된 어텐션 점수 크기 : torch.Size([1, 3, 3])
step3 어텐션 가중치 행렬 : tensor([[[0.3113, 0.3882, 0.3005],
         [0.3596, 0.2659, 0.3745],
         [0.2836, 0.4530, 0.2634]]], grad_fn=<SoftmaxBackward0>)
~~ 들의 가중치를모두 더하면 정확히 1.0 이 됩니다.
최종 문맥 벡터 크기 : torch.Size([1, 3, 2])
